# INTERSECTON

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
import warnings
warnings.filterwarnings('ignore')

from scipy.linalg import sqrtm, cholesky, cho_solve
import nonlinshrink as nls


def nlshrink_cov(Y, k=1):
    """
    Nonlinear shrinkage covariance estimation.
    
    Parameters:
    -----------
    Y : array-like, shape (n_samples, n_features)
        Data matrix
    k : int
        Number of factors (parameter for compatibility, may not be used)
    
    Returns:
    --------
    theta : array, shape (n_features, n_features)
        Shrinkage covariance estimate
    """
    # Using the non-linear-shrinkage package
    # This implements the Ledoit & Wolf nonlinear shrinkage method
    return nls.shrink_cov(Y)


def gmv_weights(Theta_hat):
    """
    Compute Global Minimum Variance (GMV) portfolio weights.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # w* = (Θ 1_p) / (1_p' Θ 1_p)
    numerator = Theta_hat @ ones_p
    denominator = ones_p @ Theta_hat @ ones_p
    
    if np.abs(denominator) < 1e-10:
        # Fallback to equal weights if precision matrix is near-singular
        return ones_p / p
    
    w_star = numerator / denominator
    
    return w_star


def mv_weights(Theta_hat, mu, target_return=0.01):
    """
    Compute Mean-Variance portfolio weights with target return.
    
    Solves the constrained optimization:
    min w' Sigma w  subject to  w' mu = target_return  and  w' 1 = 1
    
    Solution uses Lagrange multipliers with two constraints.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected returns
    target_return : float
        Target portfolio return (default: 0.01 = 1% monthly)
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute key quantities
    A = ones_p @ Theta_hat @ ones_p  # 1' Theta 1
    B = ones_p @ Theta_hat @ mu       # 1' Theta mu  
    C = mu @ Theta_hat @ mu           # mu' Theta mu
    D = A * C - B * B                  # Determinant
    
    # Check for singularity
    if np.abs(D) < 1e-10:
        # System is singular, use GMV instead
        if np.abs(A) > 1e-10:
            w_star = (Theta_hat @ ones_p) / A
            return w_star
        else:
            return ones_p / p
    
    # Compute Lagrange multipliers
    lambda1 = (C - B * target_return) / D
    lambda2 = (A * target_return - B) / D
    
    # Compute weights: w = lambda1 * Theta^{-1} 1 + lambda2 * Theta^{-1} mu
    w_star = lambda1 * (Theta_hat @ ones_p) + lambda2 * (Theta_hat @ mu)
    
    return w_star


def msr_weights(Theta_hat, mu):
    """
    Compute Maximum Sharpe Ratio portfolio weights.
    
    The maximum Sharpe ratio portfolio solves:
    max (w' mu) / sqrt(w' Sigma w)
    
    Solution (when mu represents excess returns):
    w ∝ Sigma^{-1} mu = Theta mu
    
    Then normalize so that sum(w) = 1.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected excess returns
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights (sum to 1)
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute unnormalized weights: w ∝ Theta mu
    w_unnorm = Theta_hat @ mu
    
    # Normalize to sum to 1
    weight_sum = np.sum(w_unnorm)
    
    if np.abs(weight_sum) < 1e-10:
        return ones_p / p
    
    w_star = w_unnorm / weight_sum
    
    return w_star


def load_yearly_signals(year, buys_path_template='buys_{}.csv', sells_path_template='sells_{}.csv'):
    """
    Load buy and sell signals for a specific year.
    
    Parameters:
    -----------
    year : int
        Year to load signals for
    buys_path_template : str
        Template for buys file path (use {} for year placeholder)
    sells_path_template : str
        Template for sells file path (use {} for year placeholder)
    
    Returns:
    --------
    permno_set : set
        Set of permnos in the buy and sell signals for this year
    """
    try:
        buys = pd.read_csv(buys_path_template.format(year), index_col=1)
        sells = pd.read_csv(sells_path_template.format(year), index_col=1)
        
        buys.index.name = 'permno'
        sells.index.name = 'permno'
        
        buys_index = buys.index.astype(int)
        sells_index = sells.index.astype(int)
        
        return set(buys_index.union(sells_index))
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load signals for year {year}: {e}")
        return set()


def load_finbert_signals(signals_path):
    """
    Load FinBERT monthly signals from CSV file.
    
    Parameters:
    -----------
    signals_path : str
        Path to monthly_signals.csv file
    
    Returns:
    --------
    signals_df : pd.DataFrame
        DataFrame with columns: symbol, company, year_month, signal, avg_sentiment_score
    """
    try:
        signals_df = pd.read_csv(signals_path)
        # Convert year_month to datetime (end of month)
        signals_df['date'] = pd.to_datetime(signals_df['year_month']) + pd.offsets.MonthEnd(0)
        return signals_df
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load FinBERT signals: {e}")
        return pd.DataFrame(columns=['symbol', 'company', 'year_month', 'signal', 'date'])


def get_finbert_permnos_for_date(signals_df, ticker_to_permno, date):
    """
    Get set of permnos with 'buy' or 'sell' signals for a specific date.
    
    Parameters:
    -----------
    signals_df : pd.DataFrame
        FinBERT signals dataframe
    ticker_to_permno : dict
        Mapping from ticker symbol to permno
    date : pd.Timestamp
        Date to get signals for
    
    Returns:
    --------
    permno_set : set
        Set of permnos with buy or sell signals on this date
    """
    # Get signals for this date
    date_signals = signals_df[signals_df['date'] == date]
    
    # Filter for buy and sell signals (exclude hold)
    buy_signals = date_signals[date_signals['signal'] == 'buy']
    sell_signals = date_signals[date_signals['signal'] == 'sell']
    
    # Convert tickers to permnos
    permnos = set()
    for ticker in buy_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    for ticker in sell_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    
    return permnos


def create_ticker_to_permno_mapping(df):
    """
    Create a mapping from ticker to permno from the returns dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Returns dataframe with 'ticker' and 'permno' columns
    
    Returns:
    --------
    ticker_to_permno : dict
        Mapping from ticker to permno (uses most recent permno for each ticker)
    """
    if 'ticker' not in df.columns:
        raise ValueError("DataFrame must have 'ticker' column for mapping")
    
    # Drop NaN tickers
    valid_df = df[df['ticker'].notna()].copy()
    
    # Get the most recent permno for each ticker
    ticker_to_permno = valid_df.groupby('ticker')['permno'].last().to_dict()
    
    return ticker_to_permno


def calculate_exit_transaction_cost(prev_weights_dict, prev_oos_returns_dict, 
                                    prev_gross_return, transaction_cost, verbose=False):
    """
    Calculate transaction cost when exiting the market (liquidating all positions).
    """
    if len(prev_weights_dict) == 0:
        return 0.0, 0.0, 0.0
    
    # Adjust previous weights to current period's BEGINNING
    adjusted_prev = {}
    for asset, prev_w in prev_weights_dict.items():
        if asset in prev_oos_returns_dict:
            prev_r = prev_oos_returns_dict[asset]
            if abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0
        else:
            if abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0
    
    # Turnover (Selling everything to Cash)
    turnover = sum(abs(w) for w in adjusted_prev.values())
    
    # Cost 
    tc = transaction_cost * 1.0 * turnover
    
    # Net Return is 0.0 (Cash return) - Cost
    net_return = -tc
    
    if verbose:
        print(f"  Liquidating positions | Turnover: {turnover:>6.4f} | TC: {tc:>8.6f}")
    
    return turnover, tc, net_return


def backtest_nls_finbert_combined(df, 
                                   test_start_date='2020-01-31', 
                                   test_end_date='2024-11-30',
                                   lookback_window=180,
                                   transaction_cost=0.001,
                                   buys_path_template='buys_{}.csv',
                                   sells_path_template='sells_{}.csv',
                                   finbert_signals_path=None,
                                   portfolio_types=['gmv', 'mv', 'msr'],
                                   mv_target_return=0.01,
                                   verbose=True):
    """
    Backtest NLS with year-specific buy/sell signals and FinBERT sentiment signals.
    Supports multiple portfolio types.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: permno, datadate, ticker, ret_fwd_1
    test_start_date : str
        First date for out-of-sample returns (format: 'YYYY-MM-DD')
    test_end_date : str
        Last date for out-of-sample returns (format: 'YYYY-MM-DD')
    lookback_window : int
        Number of months in rolling training window (default: 180)
    transaction_cost : float
        Proportional transaction cost (default: 0.001 = 10 bps)
    buys_path_template : str
        Template for buys file path (use {} for year placeholder)
    sells_path_template : str
        Template for sells file path (use {} for year placeholder)
    finbert_signals_path : str or None
        Path to FinBERT signals CSV file. If None, only uses buy/sell signals.
    portfolio_types : list of str
        Portfolio types to compute: 'gmv', 'mv', 'msr' (default: all three)
    mv_target_return : float
        Target return for MV portfolio (default: 0.01 = 1% monthly)
    verbose : bool
        If True, prints detailed log at each time step.
    
    Returns:
    --------
    all_results : dict
        Dictionary with keys as portfolio types, values as (results_df, metrics) tuples
    """
    # --- 1. Setup ---
    df = df.copy()
    if 'datadate' not in df.columns or 'permno' not in df.columns:
        raise ValueError("DataFrame must have 'datadate' and 'permno' columns")
    df['datadate'] = pd.to_datetime(df['datadate'])
    
    # Create ticker to permno mapping
    if verbose:
        print("Creating ticker to permno mapping...")
    ticker_to_permno = create_ticker_to_permno_mapping(df)
    if verbose:
        print(f"Mapped {len(ticker_to_permno)} unique tickers to permnos")
    
    # Load FinBERT signals if provided
    finbert_df = None
    if finbert_signals_path is not None:
        finbert_df = load_finbert_signals(finbert_signals_path)
        if verbose and len(finbert_df) > 0:
            print(f"Loaded FinBERT signals: {len(finbert_df)} monthly records")
            print(f"FinBERT signal distribution:")
            print(finbert_df['signal'].value_counts())
    
    # Get unique dates
    all_dates = sorted(df['datadate'].unique())
    
    # Convert test dates to datetime
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    # Find date indices
    try:
        test_start_idx = all_dates.index(test_start_dt)
        test_end_idx = all_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in DataFrame: {e}")
    
    if test_start_idx < lookback_window:
        raise ValueError(f"Not enough data for lookback. Test start date {test_start_date} "
                         f"requires data back to {all_dates[test_start_idx - lookback_window]}, "
                         f"but only {test_start_idx} periods are available.")
    
    # Initialize storage for each portfolio type
    portfolio_data = {ptype: {
        'returns': [],
        'dates': [],
        'weights_list': [],
        'turnover_list': [],
        'gross_returns': [],
        'prev_weights_dict': {},
        'prev_oos_returns_dict': {},
        'prev_gross_return': 0.0
    } for ptype in portfolio_types}
    
    # Cache for yearly signals
    yearly_signals_cache = {}
    
    # --- 2. Rolling Window Backtest ---
    if verbose:
        print("="*60)
        print(f"STARTING NLS+FINBERT BACKTEST: {', '.join(portfolio_types).upper()}")
        print("="*60)
        
    for t in range(test_start_idx, test_end_idx + 1):
        current_date = all_dates[t]
        current_year = current_date.year
        
        # Load signals for current year if not cached
        if current_year not in yearly_signals_cache:
            yearly_signals_cache[current_year] = load_yearly_signals(
                current_year, buys_path_template, sells_path_template
            )
        
        yearly_permnos = yearly_signals_cache[current_year]
        
        # Get FinBERT signals for current date
        finbert_permnos = set()
        if finbert_df is not None and len(finbert_df) > 0:
            finbert_permnos = get_finbert_permnos_for_date(finbert_df, ticker_to_permno, current_date)
        
        # UNION of yearly signals and FinBERT signals (with intersection fallback)
        allowed_permnos = yearly_permnos.intersection(finbert_permnos)
        if len(allowed_permnos) <= 1:
            allowed_permnos = yearly_permnos.union(finbert_permnos)
        
        # Get OOS returns FIRST
        oos_data = df[(df['datadate'] == current_date) & (df['permno'].isin(allowed_permnos))]
        oos_returns_series = oos_data.set_index('permno')['ret_fwd_1']
        oos_returns_series = oos_returns_series.dropna()
        oos_returns_dict = oos_returns_series.to_dict()
        
        # Handle early exit cases
        if len(allowed_permnos) == 0:
            if verbose:
                print(f"\n[{t - test_start_idx + 1}/{test_end_idx - test_start_idx + 1}] "
                      f"Date: {current_date.strftime('%Y-%m-%d')}")
                print(f"  ⚠ No signals, recording zero return")
            
            for ptype in portfolio_types:
                pdata = portfolio_data[ptype]
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
            continue
        
        # Define the lookback window
        window_start_date = all_dates[t - lookback_window]
        window_end_date = all_dates[t - 1]
        
        # Get training data
        train_data = df[(df['datadate'] >= window_start_date) & 
                        (df['datadate'] <= window_end_date) &
                        (df['permno'].isin(allowed_permnos))]
        
        # Pivot to get returns matrix
        returns_pivot = train_data.pivot(index='datadate', columns='permno', values='ret_fwd_1')
        window_dates = all_dates[t - lookback_window : t]
        returns_pivot = returns_pivot.reindex(index=window_dates)
        
        # Filter assets with NaNs
        nan_assets = returns_pivot.columns[returns_pivot.isna().any()]
        filtered_pivot = returns_pivot.drop(columns=nan_assets)
        
        current_assets = filtered_pivot.columns.tolist()
        Y = filtered_pivot.values
        n_train, p_current = Y.shape
    
        if verbose:
            print(f"\n[{t - test_start_idx + 1}/{test_end_idx - test_start_idx + 1}] "
                  f"Date: {current_date.strftime('%Y-%m-%d')} | Year: {current_year}")
            print(f"  Window: {window_start_date.strftime('%Y-%m-%d')} to "
                  f"{window_end_date.strftime('%Y-%m-%d')}")
            print(f"  Yearly: {len(yearly_permnos)} | FinBERT: {len(finbert_permnos)} | "
                  f"Combined: {len(allowed_permnos)} | Assets: {p_current}")
    
        # Check for valid data
        if n_train < lookback_window or p_current < 2:
            if verbose:
                print(f"  ⚠ Insufficient data (n={n_train}, p={p_current}), recording zero return")
            
            for ptype in portfolio_types:
                pdata = portfolio_data[ptype]
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
            continue
        
        try:
            # Demean the returns
            Y_bar = Y.mean(axis=0)
            Y_star = Y - Y_bar
            
            if verbose:
                print(f"  Running NLS...")
            Sigma_hat = nlshrink_cov(Y_star)
            
            # Invert to get precision matrix
            try:
                Theta_hat = np.linalg.inv(Sigma_hat)
            except np.linalg.LinAlgError:
                if verbose:
                    print(f"  ⚠ Covariance matrix singular, using pseudo-inverse")
                Theta_hat = np.linalg.pinv(Sigma_hat)
            
            # Compute weights for each portfolio type
            weights_dict_by_type = {}
            
            if 'gmv' in portfolio_types:
                if verbose:
                    print(f"  Computing GMV weights...")
                w_gmv = gmv_weights(Theta_hat)
                weights_dict_by_type['gmv'] = {asset: w_gmv[i] for i, asset in enumerate(current_assets)}
            
            if 'mv' in portfolio_types or 'msr' in portfolio_types:
                # Compute mean returns for MV and MSR
                mu = Y_bar
                
                if 'mv' in portfolio_types:
                    if verbose:
                        print(f"  Computing MV weights (target={mv_target_return})...")
                    w_mv = mv_weights(Theta_hat, mu, target_return=mv_target_return)
                    weights_dict_by_type['mv'] = {asset: w_mv[i] for i, asset in enumerate(current_assets)}
                
                if 'msr' in portfolio_types:
                    if verbose:
                        print(f"  Computing MSR weights...")
                    w_msr = msr_weights(Theta_hat, mu)
                    weights_dict_by_type['msr'] = {asset: w_msr[i] for i, asset in enumerate(current_assets)}
            
        except Exception as e:
            if verbose:
                print(f"  ✗ Error: {e}")
                print(f"  Recording zero return for all portfolios")
            
            for ptype in portfolio_types:
                pdata = portfolio_data[ptype]
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
            continue

        # Process each portfolio type separately
        for ptype in portfolio_types:
            pdata = portfolio_data[ptype]
            new_weights_dict = weights_dict_by_type[ptype]
            
            # Normalize weights to sum to 1
            weight_sum = sum(new_weights_dict.values())
            if weight_sum > 1e-10:
                new_weights_dict = {k: v/weight_sum for k, v in new_weights_dict.items()}
            else:
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
                continue
            
            # Find common assets
            common_assets = set(new_weights_dict.keys()) & set(oos_returns_dict.keys())
            
            if len(common_assets) == 0:
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
                continue
            
            # Filter and renormalize
            common_weights = {a: new_weights_dict[a] for a in common_assets}
            common_weight_sum = sum(common_weights.values())
            if common_weight_sum > 1e-10:
                common_weights = {k: v/common_weight_sum for k, v in common_weights.items()}
            else:
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
                continue
            
            # Compute gross return
            gross_return = sum(common_weights[a] * oos_returns_dict[a] for a in common_assets)
            
            # Sanity check
            if np.isnan(gross_return) or np.isinf(gross_return):
                turnover, tc, net_return = calculate_exit_transaction_cost(
                    pdata['prev_weights_dict'], 
                    pdata['prev_oos_returns_dict'], 
                    pdata['prev_gross_return'], 
                    transaction_cost,
                    verbose=False
                )
                
                pdata['returns'].append(net_return)
                pdata['dates'].append(current_date)
                pdata['weights_list'].append({})
                pdata['turnover_list'].append(turnover)
                pdata['gross_returns'].append(0.0)
                pdata['prev_weights_dict'] = {}
                pdata['prev_oos_returns_dict'] = {}
                pdata['prev_gross_return'] = 0.0
                continue
            
            # Calculate transaction costs
            if len(pdata['prev_weights_dict']) > 0:
                # Adjust previous weights for returns
                adjusted_prev = {}
                for asset, prev_w in pdata['prev_weights_dict'].items():
                    if asset in pdata['prev_oos_returns_dict']:
                        prev_r = pdata['prev_oos_returns_dict'][asset]
                        if abs(1 + pdata['prev_gross_return']) > 1e-6:
                            adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + pdata['prev_gross_return'])
                        else:
                            adjusted_prev[asset] = 0.0
                    else:
                        if abs(1 + pdata['prev_gross_return']) > 1e-6:
                            adjusted_prev[asset] = prev_w / (1 + pdata['prev_gross_return'])
                        else:
                            adjusted_prev[asset] = 0.0
                
                # Calculate turnover
                all_assets = set(adjusted_prev.keys()) | set(common_weights.keys())
                turnover = 0.0
                for asset in all_assets:
                    old_w = adjusted_prev.get(asset, 0.0)
                    new_w = common_weights.get(asset, 0.0)
                    turnover += abs(new_w - old_w)
                
                tc = transaction_cost * (1 + gross_return) * turnover
            else:
                # First period
                turnover = sum(abs(w) for w in common_weights.values())
                tc = transaction_cost * (1 + gross_return) * turnover
            
            # Net return
            net_return = gross_return - tc
            
            # Store results
            pdata['returns'].append(net_return)
            pdata['dates'].append(current_date)
            pdata['weights_list'].append(common_weights.copy())
            pdata['turnover_list'].append(turnover)
            pdata['gross_returns'].append(gross_return)
            pdata['prev_weights_dict'] = common_weights.copy()
            pdata['prev_oos_returns_dict'] = {a: oos_returns_dict[a] for a in common_assets}
            pdata['prev_gross_return'] = gross_return
        
        if verbose:
            # Print summary for all portfolios
            for ptype in portfolio_types:
                pdata = portfolio_data[ptype]
                if len(pdata['returns']) > 0:
                    last_return = pdata['returns'][-1]
                    last_gross = pdata['gross_returns'][-1] if len(pdata['gross_returns']) > 0 else 0
                    last_turnover = pdata['turnover_list'][-1] if len(pdata['turnover_list']) > 0 else 0
                    last_tc = last_gross - last_return
                    print(f"  {ptype.upper()}: Gross={last_gross:>8.5f} | TO={last_turnover:>6.4f} | "
                          f"TC={last_tc:>8.6f} | Net={last_return:>8.5f}")

    if verbose:
        print("\n" + "="*60)
        print("BACKTEST COMPLETE")
        print("="*60)
    
    # --- 4. Compile Results for Each Portfolio ---
    all_results = {}
    
    for ptype in portfolio_types:
        pdata = portfolio_data[ptype]
        
        results_df = pd.DataFrame({
            'date': pdata['dates'],
            'portfolio_return': pdata['returns'],
            'portfolio_gross_return': pdata['gross_returns'],
            'portfolio_weights': pdata['weights_list'],
            'portfolio_turnover': pdata['turnover_list']
        })
        results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1
        
        # Compute metrics
        if len(pdata['returns']) > 0:
            mean_return = np.mean(pdata['returns'])
            variance = np.var(pdata['returns'], ddof=1)
            sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
            
            # Annualized metrics (monthly data)
            annual_return = mean_return * 12
            annual_volatility = np.sqrt(variance * 12)
            annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
            
            metrics = {
                'mean_return': mean_return,
                'variance': variance,
                'sharpe_ratio': sharpe_ratio,
                'annual_return': annual_return,
                'annual_volatility': annual_volatility,
                'annual_sharpe_ratio': annual_sharpe,
                'total_return': results_df['cumulative_return'].iloc[-1],
                'avg_turnover': np.mean(pdata['turnover_list']),
                'n_periods': len(pdata['returns']),
                'n_zero_periods': sum(1 for r in pdata['returns'] if r == 0)
            }
        else:
            metrics = {
                'mean_return': 0,
                'variance': 0,
                'sharpe_ratio': 0,
                'annual_return': 0,
                'annual_volatility': 0,
                'annual_sharpe_ratio': 0,
                'total_return': 0,
                'avg_turnover': 0,
                'n_periods': 0,
                'n_zero_periods': 0
            }
        
        all_results[ptype] = (results_df, metrics)
    
    return all_results

In [3]:
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

In [4]:
all_results = backtest_nls_finbert_combined(
    df,
    test_start_date='2023-11-30',
    test_end_date='2024-04-30',
    lookback_window=180,
    transaction_cost=0.001,
    buys_path_template='../AI Portfolio Selection/novy_marx_buys_{}.csv',
    sells_path_template='../AI Portfolio Selection/novy_marx_sells_{}.csv',
    finbert_signals_path='../examples/monthly_signals_decay.csv',  # Your FinBERT signals
    verbose=True
)

# Access individual results
gmv_results, gmv_metrics = all_results['gmv']
mv_results, mv_metrics = all_results['mv']
msr_results, msr_metrics = all_results['msr']

# Compare performance
print(f"GMV Annual Sharpe: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MV Annual Sharpe: {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"MSR Annual Sharpe: {msr_metrics['annual_sharpe_ratio']:.4f}")

Creating ticker to permno mapping...
Mapped 1664 unique tickers to permnos
Loaded FinBERT signals: 24780 monthly records
FinBERT signal distribution:
signal
hold    23840
sell      529
buy       411
Name: count, dtype: int64
STARTING NLS+FINBERT BACKTEST: GMV, MV, MSR

[1/6] Date: 2023-11-30 | Year: 2023
  Window: 2008-11-30 to 2023-10-31
  Yearly: 300 | FinBERT: 35 | Combined: 16 | Assets: 13
  Running NLS...
  Computing GMV weights...
  Computing MV weights (target=0.01)...
  Computing MSR weights...
  GMV: Gross= 0.01098 | TO=1.1528 | TC=0.001166 | Net= 0.00982
  MV: Gross= 0.00549 | TO=1.1826 | TC=0.001189 | Net= 0.00430
  MSR: Gross= 0.03190 | TO=1.5910 | TC=0.001642 | Net= 0.03026

[2/6] Date: 2023-12-31 | Year: 2023
  Window: 2008-12-31 to 2023-11-30
  Yearly: 300 | FinBERT: 36 | Combined: 23 | Assets: 16
  Running NLS...
  Computing GMV weights...
  Computing MV weights (target=0.01)...
  Computing MSR weights...
  GMV: Gross=-0.01052 | TO=1.6849 | TC=0.001667 | Net=-0.01218
  

In [5]:
print(f"\n GMV")
print(f"Annualized Sharpe Ratio: {gmv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {gmv_metrics['mean_return']*12:.4f}")
print(f"Variance: {gmv_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {gmv_metrics['avg_turnover']:.4f}")

print(f"\n MV")
print(f"Annualized Sharpe Ratio: {mv_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {mv_metrics['mean_return']*12:.4f}")
print(f"Variance: {mv_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {mv_metrics['avg_turnover']:.4f}")

print(f"\n MSR")
print(f"Annualized Sharpe Ratio: {msr_metrics['annual_sharpe_ratio']:.4f}")
print(f"Mean Return: {msr_metrics['mean_return']*12:.4f}")
print(f"Variance: {msr_metrics['variance']*12:.4f}")
print(f"Avg Turnover: {msr_metrics['avg_turnover']:.4f}")


 GMV
Annualized Sharpe Ratio: 1.5171
Mean Return: 0.2121
Variance: 0.0195
Avg Turnover: 1.8968

 MV
Annualized Sharpe Ratio: 1.0169
Mean Return: 0.1523
Variance: 0.0224
Avg Turnover: 1.9622

 MSR
Annualized Sharpe Ratio: 4.2105
Mean Return: 0.4430
Variance: 0.0111
Avg Turnover: 2.6890
